# Predict all crops with the saved Extra Trees pipeline

This local Jupyter notebook loads the fitted Extra Trees pipeline and `future_horizon_features.csv` from this project, predicts every crop/location/H1-H12 row, and saves full and date-filtered forecasts under `artifacts/predictions`. It never creates hypothetical features or replaces missing future context with zero.

In [1]:
# Run this only if the local Jupyter environment is missing dependencies.
# %pip install -r ../requirements.txt

## Configure local project files

The model is read from `artifacts/models`, future features from `data/processed`, and predictions are written to `artifacts/predictions`. The project root is detected whether Jupyter starts in the project root or the `notebooks` folder.

In [2]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn

def find_project_root(start_path):
    start_path = start_path.resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "pipeline_config.json").exists():
            return candidate
    raise FileNotFoundError(
        "Project root not found. Start Jupyter inside the demand-forecasting project."
    )

PROJECT_ROOT = find_project_root(Path.cwd())
MODEL_FILE = PROJECT_ROOT / "artifacts" / "models" / "global_etr_pipeline.joblib"
FUTURE_DATA_FILE = PROJECT_ROOT / "data" / "processed" / "future_horizon_features.csv"
METADATA_FILE = PROJECT_ROOT / "artifacts" / "reports" / "etr_training_metadata.json"
OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "predictions"

for required_file in [MODEL_FILE, FUTURE_DATA_FILE, METADATA_FILE]:
    if not required_file.exists():
        raise FileNotFoundError(f"Required file not found: {required_file}")

training_metadata = json.loads(METADATA_FILE.read_text(encoding="utf-8"))
trained_sklearn_version = training_metadata["versions"]["scikit_learn"]
if sklearn.__version__ != trained_sklearn_version:
    raise RuntimeError(
        f"scikit-learn version mismatch: model={trained_sklearn_version}, "
        f"local={sklearn.__version__}. Run `%pip install scikit-learn=={trained_sklearn_version}` "
        "and then restart the Jupyter kernel."
    )

print(f"Project root: {PROJECT_ROOT}")
print(f"Model: {MODEL_FILE}")
print(f"Future features: {FUTURE_DATA_FILE}")

Project root: C:\demand-forecasting
Model: C:\demand-forecasting\artifacts\models\global_etr_pipeline.joblib
Future features: C:\demand-forecasting\data\processed\future_horizon_features.csv


## Load and validate future features

Prediction stops when required columns or future context values are missing. This prevents plausible-looking forecasts based on invented inputs.

In [3]:
model_pipeline = joblib.load(MODEL_FILE)
future_data = pd.read_csv(
    FUTURE_DATA_FILE,
    parse_dates=["forecast_origin_month", "target_month"],
)

feature_columns = list(model_pipeline.feature_names_in_)
missing_features = [column for column in feature_columns if column not in future_data.columns]
if missing_features:
    raise ValueError(f"Future dataset is missing model features: {missing_features}")

required_future_context = [
    "target_public_holiday_count",
    "target_cny",
    "target_hari_raya",
    "target_gawai",
    "target_christmas",
    "target_major_festival",
    "target_month_normal_temperature",
    "target_month_normal_precipitation",
]
missing_context = future_data[required_future_context].isna().sum()
missing_context = missing_context[missing_context > 0]
if not missing_context.empty:
    raise ValueError(
        "Future context is incomplete. Regenerate future_horizon_features.csv. "
        "Missing values: " + str(missing_context.to_dict())
    )

expected_rows = future_data[["crop", "location"]].drop_duplicates().shape[0] * 12
if len(future_data) != expected_rows:
    raise ValueError(f"Expected {expected_rows} H1-H12 rows but found {len(future_data)}.")

print(f"Validated {len(future_data):,} prediction rows.")
print(f"Target range: {future_data.target_month.min().date()} to {future_data.target_month.max().date()}")

Validated 540 prediction rows.
Target range: 2026-01-01 to 2026-12-01


## Predict all crop, location, and horizon rows

In [4]:
X_future = future_data[feature_columns].copy()
predicted_demand = model_pipeline.predict(X_future)

future_data["predicted_demand_kg"] = np.maximum(predicted_demand, 0)
forecast_output = future_data[
    [
        "forecast_origin_month",
        "target_month",
        "forecast_horizon",
        "crop",
        "location",
        "predicted_demand_kg",
    ]
].sort_values(["target_month", "crop", "location"]).reset_index(drop=True)

display(forecast_output.head(30))

,forecast_origin_month,target_month,forecast_horizon,crop,location,predicted_demand_kg
0,2025-12-01,2026-01-01,1,Bittergourd,Kuching,2719.763333
1,2025-12-01,2026-01-01,1,Bittergourd,Miri,1977.254333
2,2025-12-01,2026-01-01,1,Bittergourd,Sibu,1731.305667
3,2025-12-01,2026-01-01,1,Cabbage,Kuching,5693.588000
4,2025-12-01,2026-01-01,1,Cabbage,Miri,3524.673167
5,2025-12-01,2026-01-01,1,Cabbage,Sibu,3482.407667
6,2025-12-01,2026-01-01,1,Choysum,Kuching,3831.648000
7,2025-12-01,2026-01-01,1,Choysum,Miri,2911.071667
8,2025-12-01,2026-01-01,1,Choysum,Sibu,2832.173000
9,2025-12-01,2026-01-01,1,Cucumber,Kuching,4455.758000


## Select a target date range

The example selects October through November 2026. Change the dates without changing model features.

In [5]:
SELECTED_START = pd.Timestamp("2026-10-01")
SELECTED_END = pd.Timestamp("2026-11-01")

selected_forecast = forecast_output[
    forecast_output["target_month"].between(SELECTED_START, SELECTED_END)
].copy()

if selected_forecast.empty:
    raise ValueError("No forecast rows fall within the selected target range.")

print(f"Selected rows: {len(selected_forecast):,}")
display(selected_forecast)

Selected rows: 90


,forecast_origin_month,target_month,forecast_horizon,crop,location,predicted_demand_kg
405,2025-12-01,2026-10-01,10,Bittergourd,Kuching,2275.616333
406,2025-12-01,2026-10-01,10,Bittergourd,Miri,1084.400333
407,2025-12-01,2026-10-01,10,Bittergourd,Sibu,1390.202667
408,2025-12-01,2026-10-01,10,Cabbage,Kuching,3594.171000
409,2025-12-01,2026-10-01,10,Cabbage,Miri,2222.026167
...,...,...,...,...,...,...
490,2025-12-01,2026-11-01,11,Sweetcorn,Miri,3483.377833
491,2025-12-01,2026-11-01,11,Sweetcorn,Sibu,3685.248000
492,2025-12-01,2026-11-01,11,Tomato,Kuching,4012.846333
493,2025-12-01,2026-11-01,11,Tomato,Miri,2647.707667


## Save full and selected forecasts

In [6]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

full_name = "etr_future_forecast_h1_h12.csv"
selected_name = "etr_forecast_2026_10_to_2026_11.csv"

forecast_output.to_csv(OUTPUT_DIR / full_name, index=False)
selected_forecast.to_csv(OUTPUT_DIR / selected_name, index=False)

print(f"Full forecast: {OUTPUT_DIR / full_name}")
print(f"Selected forecast: {OUTPUT_DIR / selected_name}")

Full forecast: C:\demand-forecasting\artifacts\predictions\etr_future_forecast_h1_h12.csv
Selected forecast: C:\demand-forecasting\artifacts\predictions\etr_forecast_2026_10_to_2026_11.csv
